In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV ,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import joblib
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.neighbors import KNeighborsClassifier



In [2]:
!pip install xgboost


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:

x_train=np.load('./data/embeddings/train_embeddings.npy')
x_test=np.load('./data/embeddings/test_embeddings.npy')


In [4]:
x_train

array([[ 0.07367804, -0.02631131,  0.0390068 , ..., -0.07857121,
         0.05455049,  0.07414829],
       [-0.0068568 , -0.06906828, -0.04574034, ..., -0.06036085,
         0.04355335,  0.04123302],
       [-0.01750738, -0.03781661,  0.02444598, ..., -0.11092535,
        -0.02253906,  0.04098497],
       ...,
       [-0.02288846,  0.01629856, -0.03647592, ...,  0.03625312,
         0.02009865,  0.04081287],
       [-0.03029878, -0.02274068, -0.00463907, ...,  0.01157963,
        -0.01234237,  0.02648924],
       [-0.03888904, -0.05788031, -0.02970093, ...,  0.03459368,
         0.02085046, -0.06742197]], shape=(120000, 384), dtype=float32)

In [5]:
metadata_train=pd.read_csv("./data/metadata/train_metadata.csv")
metadata_test=pd.read_csv("./data/metadata/test_metadata.csv")
y_test=metadata_test["label"].values
y_train=metadata_train["label"].values



In [6]:
import numpy as np

print(y_train)


[2 2 2 ... 1 1 1]


In [7]:
models={
    "SVM":LinearSVC(max_iter=5000),
    "xgboost":xgb.XGBClassifier(n_estimators=500),
    "LogisticRegression":LogisticRegression(max_iter=1000),
    "knn" : KNeighborsClassifier(n_neighbors=5)
}

In [ ]:
for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    joblib.dump(model,f'./models/{name}_model.joblib')
    print(f"=== {name} ===")
    print("Model Accuracy:\n")
    print(f"accuracy_score:{accuracy_score(y_test,y_pred)}\n")

    # print(f'roc_auc_score  :{roc_auc_score(y_test, y_pred[:, 1])}\n')
    print("Confusion Matrix:\n")

    print(f"confusion_matrix :{confusion_matrix(y_test,y_pred)}")
    print(f"classification_report:{classification_report(y_test,y_pred)}")
    print("\n")


grid_search = GridSearchCV(
    estimator=model,       # modèle à optimiser
    param_grid=param_grid, # grille de paramètres
    scoring='accuracy',    # critère de performance ('roc_auc', 'f1', etc.)
    cv=5,                  # nombre de folds pour la cross-validation
    n_jobs=-1,             # utiliser tous les cœurs
    verbose=2,             # afficher les logs
    refit=True             # si True, entraîne le meilleur modèle sur toutes les données
)


In [ ]:
model=xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss'
    )

param_grid={
    'n_estimators': [50,100],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.01],
    'subsample': [0.8, 1.0]
}


grid_search= GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    refit=True,
    cv=5,
    n_jobs=-1   
)
grid_search.fit(x_train,y_train)
best_model=grid_search.best_estimator_
y_pred=best_model.predict(x_test)

print(f"accuracy_score:{accuracy_score(y_test,y_pred)}\n")

print("Confusion Matrix:\n")


print(f"confusion_matrix :  {confusion_matrix(y_test,y_pred)}")

print(f"classification_report: { classification_report(y_test,y_pred)}")

print("\n")

joblib.dump(best_model,"./models/best_model.joblib")